# PyMIF beginner notebook 02 - ArrayManager from NumPy or Dask arrays

Use `ArrayManager` when your image is already in Python as a NumPy or Dask array. This is the cleanest way to create PyMIF-compatible data without a microscope-specific reader.

The most common beginner mistakes are:

- forgetting to set `metadata["axes"]`,
- giving `scales` in the wrong order,
- using `TCZYX` metadata for an array that is actually `ZYX` or `YX`,
- not setting `data_type` for labels.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

## Example 1: a standard 5D intensity image in TCZYX order

In [ ]:
rng = np.random.default_rng(10)
raw = rng.integers(0, 4096, size=(2, 3, 5, 96, 96), dtype=np.uint16)

metadata = {
    "name": "array_tczyx_example",
    "axes": "tczyx",
    "size": [raw.shape],
    "chunksize": [(1, 1, 2, 48, 48)],
    "scales": [(1.5, 0.25, 0.25)],
    "units": ("micrometer", "micrometer", "micrometer"),
    "time_increment": 30.0,
    "time_increment_unit": "second",
    "channel_names": ["DAPI", "GFP", "RFP"],
    "channel_colors": ["0000FF", "00FF00", "FF0000"],
    "dtype": "uint16",
    "data_type": "intensity",
}

manager = mm.ArrayManager(raw, metadata, chunks=metadata["chunksize"][0])
summarize_manager(manager, "TCZYX ArrayManager")

## Build a pyramid

`build_pyramid` uses nearest-neighbor downsampling. For fluorescence volumes, a common choice is `(1, 2, 2)`: keep Z unchanged and downsample Y and X.

In [ ]:
manager.build_pyramid(num_levels=3, downscale_factor=(1, 2, 2))
summarize_manager(manager, "After building a 3-level pyramid")

## Example 2: a 2D image in YX order

The current API is axis-aware. You do not need to fake `T`, `C`, or `Z` axes.

In [ ]:
image_2d = rng.integers(0, 255, size=(128, 192), dtype=np.uint8)
metadata_2d = {
    "name": "brightfield_yx",
    "axes": "yx",
    "size": [image_2d.shape],
    "chunksize": [(64, 64)],
    "scales": [(0.65, 0.65)],
    "units": ("micrometer", "micrometer"),
    "dtype": "uint8",
    "data_type": "intensity",
}

manager_2d = mm.ArrayManager(image_2d, metadata_2d, chunks=metadata_2d["chunksize"][0])
manager_2d.build_pyramid(num_levels=2, downscale_factor=2)
summarize_manager(manager_2d, "YX ArrayManager")

## Example 3: labels in TZYX order

Labels should use an integer dtype and `data_type="label"`. Most labels do not have a channel axis.

In [ ]:
labels = np.zeros((2, 5, 96, 96), dtype=np.uint32)  # T, Z, Y, X
labels[:, :, 20:55, 25:60] = 1
labels[:, :, 60:80, 65:85] = 2

label_metadata = {
    "name": "nuclei",
    "axes": "tzyx",
    "size": [labels.shape],
    "chunksize": [(1, 2, 48, 48)],
    "scales": [(1.5, 0.25, 0.25)],
    "units": ("micrometer", "micrometer", "micrometer"),
    "time_increment": 30.0,
    "time_increment_unit": "second",
    "dtype": "uint32",
    "data_type": "label",
    "channel_names": [],
    "channel_colors": [],
}

label_manager = mm.ArrayManager(labels, label_metadata, chunks=label_metadata["chunksize"][0])
label_manager.build_pyramid(num_levels=3, downscale_factor=(1, 2, 2))
summarize_manager(label_manager, "TZYX label ArrayManager")

## Write array data to OME-Zarr

The default modern output is NGFF v0.5 with Zarr v3. Use explicit `ngff_version` and `zarr_format` when you need reproducibility.

In [ ]:
out = clean_path(OUTPUT_DIR / "array_manager_raw.zarr")
manager.to_zarr(out, ngff_version="0.5", zarr_format=3)
print("Wrote", out)

# A label-only OME-Zarr is possible, but this is not the same as labels inside an image Zarr.
label_out = clean_path(OUTPUT_DIR / "label_only.zarr")
label_manager.to_zarr(label_out, ngff_version="0.5", zarr_format=3, data_type="label")
print("Wrote", label_out)